In [ ]:
import tifffile as tiff
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def display_viirs(img):
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

def display_sentinel(img):
    fig, axs = plt.subplots(3, 4, figsize=(15, 10))
    for i, ax in zip(range(12), axs.flatten()):
        ax.imshow(img[:, :, i])
        ax.set_title(f'B{i+1}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

def print_row_info(row, DataPath):
    print("Processing data_id: " + str(row.data_id))
    print("Construction cost per m2 usd: " + str(row.construction_cost_per_m2_usd))
    print("Year: " + str(row.year) + " | Quarter: " + str(row.quarter_label))
    print("Country: " + str(row.country) + " | geolocation: " + str(row.geolocation_name))
    print("Economic data:")
    print("  deflated_gdp_usd: " + str(row.deflated_gdp_usd))
    print("  us_cpi: " + str(row.us_cpi))
    print("  developed_country: " + str(row.developed_country))
    print("  economic_classification: " + str(row.region_economic_classification))
    print("Infrastructure access:")
    print("  landlocked: " + str(row.landlocked))
    print("  access_to_airport: " + str(row.access_to_airport))
    print("  access_to_port: " + str(row.access_to_port))
    print("  access_to_highway: " + str(row.access_to_highway))
    print("  access_to_railway: " + str(row.access_to_railway))
    print("Geographical data:")
    print("  straight_distance_to_capital_km: " + str(row.straight_distance_to_capital_km))
    print("  seismic_hazard_zone: " + str(row.seismic_hazard_zone))
    print("  flood_risk_class: " + str(row.flood_risk_class))
    print("  tropical_cyclone_wind_risk: " + str(row.tropical_cyclone_wind_risk))
    print("  tornadoes_wind_risk: " + str(row.tornadoes_wind_risk))
    print("  koppen_climate_zone: " + str(row.koppen_climate_zone))
    print("Image files:")

    imgPath = DataPath / "train_composite"
    
    viirs = tiff.imread(imgPath / row.viirs_tiff_file_name)
    print("Viirs: " + str(viirs.shape))
    display_viirs(viirs)

    sentinel = tiff.imread(imgPath / row.sentinel2_tiff_file_name)
    print("Sentinel: " + str(sentinel.shape))
    display_sentinel(sentinel)
    print()


In [ ]:
DataPath = Path("..") / "Training data"

train_tabular = pd.read_csv(DataPath / "train_tabular.csv")
#print(train_tabular.head())
print(f"Tabular shape: {train_tabular.shape}")

print(train_tabular.columns)
print()


for row in train_tabular.head().itertuples(index=False):
    print_row_info(row, DataPath)

In [ ]:
japan = train_tabular[train_tabular['country'] == 'Japan']
philipines = train_tabular[train_tabular['country'] == 'Philippines']

datasets = [("Japan", japan), ("Philippines", philipines)]

# Get numeric columns only
numeric_columns = japan.select_dtypes(include=[np.number]).columns.tolist()
numeric_columns.remove('year')


# Create a separate plot for each numeric column
for col in numeric_columns:
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    
    for idx, (key, dataset) in enumerate(datasets):
        axs[idx].hist(dataset[col].dropna(), bins=30, color='blue', alpha=0.7)
        axs[idx].set_title(f'{key} - {col}')
        axs[idx].set_xlabel(col)
        axs[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

# Get non numeric columns only
non_numeric_columns = japan.select_dtypes(exclude=[np.number]).columns.tolist()

non_numeric_columns.remove('viirs_tiff_file_name')
non_numeric_columns.remove('sentinel2_tiff_file_name')
non_numeric_columns.remove('data_id')
non_numeric_columns.remove('country')

# Create a separate plot for each non-numeric column
for col in non_numeric_columns:
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    
    for idx, (key, dataset) in enumerate(datasets):
        dataset[col].value_counts().plot(kind='bar', ax=axs[idx], color='blue', alpha=0.7)
        axs[idx].set_title(f'{key} - {col}')
        axs[idx].set_xlabel(col)
        axs[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()